In [28]:
#!/usr/bin/env python3

import sys
import os
import pandas as pd
import numpy as np
import time
import scipy

# scipy version compatibility patch
minimal_patch = [
    'zeros_like', 'ones_like', 'argsort', 'array', 'arange',
    'sum', 'mean', 'var', 'std', 'min', 'max',
    'where', 'unique', 'sort', 'clip', 'dot',
    'exp', 'log', 'sqrt', 'square', 'abs', 'sign',
    'sin', 'cos', 'pi', 'inf', 'linspace', 'logspace',
    'log10', 'ones', 'zeros', 'eye',
    'transpose', 'reshape', 'flatten', 'ravel',
    'concatenate', 'hstack', 'vstack',
    'argmin', 'argmax', 'nonzero', 'isfinite', 'isnan', 'isinf'
]
for func_name in minimal_patch:
    if not hasattr(scipy, func_name) and hasattr(np, func_name):
        setattr(scipy, func_name, getattr(np, func_name))

In [ ]:
import SpatialDE
import NaiveDE

def main():
    if len(sys.argv) < 2:
        print("Error: SI value not provided")
        print("Usage: python p.spatialde2.py <SI_value>")
        sys.exit(1)
    
    si = float(sys.argv[1])
    sit = round(si, 2)
    
    print(f"Starting SpatialDE analysis for SI: {sit}")
    
    dpath = "data/sim.generated"
    rpath = "data/bhmk.res/p/SPATIALDE"
    
    os.makedirs(rpath, exist_ok=True)
    
    try:
        count_path = os.path.join(dpath, "sim.prog", f"sim_prog_SI{sit}.csv")
        perm_path = os.path.join(dpath, "sim.prog", f"perm_prog_SI{sit}.csv")
        gt_path = os.path.join(dpath, "sim.prog", f"GT_SI{sit}.csv")
        coord_path = os.path.join(dpath, "coord.csv")
        
        if not os.path.exists(count_path):
            print(f"Error: File not found - {count_path}")
            sys.exit(1)
            
        counts = pd.read_csv(count_path, index_col=0)
        perm = pd.read_csv(perm_path, index_col=0)
        g = pd.read_csv(gt_path)
        coord = pd.read_csv(coord_path, index_col=0)
        
        # Calculate total_counts from ORIGINAL raw counts BEFORE any processing
        counts_raw = counts.copy()
        coord['total_counts'] = counts_raw.sum(axis=0)
        
        t0 = time.time()
        
        # NaiveDE normalization
        counts_norm = NaiveDE.stabilize(counts_raw)
        
        # SpatialDE analysis
        resid_expr = NaiveDE.regress_out(coord, counts_norm, 'np.log(total_counts)').T
        X = coord[['X1', 'X2']].values
        ressig = SpatialDE.run(X, resid_expr)
        
        t1 = time.time() - t0
        
        # Run on permuted data
        perm_norm = NaiveDE.stabilize(perm)
        coord_perm = coord.copy()
        coord_perm['total_counts'] = perm.sum(axis=0)
        resid_expr_perm = NaiveDE.regress_out(coord_perm, perm_norm, 'np.log(total_counts)').T
        resperm = SpatialDE.run(X, resid_expr_perm)
        
        gs = g['gs'].values
        gc = g['gc'].values
        thr = 0.05
        svgsim = ressig[ressig["qval"] < thr]["g"].values
        svgprm = resperm[resperm["qval"] < thr]["g"].values
        
        res = pd.DataFrame({
            'ID': ["Power_sim", "FPR_sim", "Power_prm", "FPR_prm"],
            'SPATIALDE': [
                len(np.intersect1d(gs, svgsim)) / len(gs) if len(gs) > 0 else 0,
                len(np.intersect1d(gc, svgsim)) / len(gc) if len(gc) > 0 else 0,
                len(np.intersect1d(gs, svgprm)) / len(gs) if len(gs) > 0 else 0,
                len(np.intersect1d(gc, svgprm)) / len(gc) if len(gc) > 0 else 0
            ],
            'SI': [sit] * 4,
            'total_time': [t1] * 4,
            'mapping_time': [np.nan] * 4
        })
        
        res.to_csv(os.path.join(rpath, f"res_spatialde_SI{sit}.csv"), index=False)
        
    except Exception as e:
        print(f"Error during SpatialDE analysis: {str(e)}")
        sys.exit(1)

if __name__ == "__main__":
    main()

# Test SpatialDE Analysis Step by Step

각 단계별로 테스트해보기 위한 코드블럭들입니다.

In [4]:
# Step 1: 파라미터 설정
si = 0.47  # 여기에 원하는 SI 값 입력
sit = round(si, 2)

print(f"Testing SpatialDE analysis for SI: {sit}")

# 경로 설정
dpath = "../data/sim.generated"
rpath = "../data/bhmk.res/p/SPATIALDE"

os.makedirs(rpath, exist_ok=True)

Testing SpatialDE analysis for SI: 0.47


In [25]:
# Step 2: 데이터 로딩
count_path = os.path.join(dpath, "sim.prog", f"sim_prog_SI{sit}.csv")
perm_path = os.path.join(dpath, "sim.prog", f"perm_prog_SI{sit}.csv")
gt_path = os.path.join(dpath, "sim.prog", f"GT_SI{sit}.csv")
coord_path = os.path.join(dpath, "coord.csv")

print("Loading files:")
print(f"Count: {count_path}")
print(f"Perm: {perm_path}")
print(f"GT: {gt_path}")
print(f"Coord: {coord_path}")

if not os.path.exists(count_path):
    print(f"Error: File not found - {count_path}")
else:
    counts = pd.read_csv(count_path, index_col=0)
    perm = pd.read_csv(perm_path, index_col=0)
    g = pd.read_csv(gt_path)
    coord = pd.read_csv(coord_path, index_col=0)
    
    print(f"Counts shape: {counts.shape}")
    print(f"Perm shape: {perm.shape}")
    print(f"GT shape: {g.shape}")
    print(f"Coord shape: {coord.shape}")
    print("Data loaded successfully!")

Loading files:
Count: ../data/sim.generated/sim.prog/sim_prog_SI0.47.csv
Perm: ../data/sim.generated/sim.prog/perm_prog_SI0.47.csv
GT: ../data/sim.generated/sim.prog/GT_SI0.47.csv
Coord: ../data/sim.generated/coord.csv
Counts shape: (2000, 3852)
Perm shape: (2000, 3852)
GT shape: (1000, 2)
Coord shape: (3852, 2)
Data loaded successfully!
Counts shape: (2000, 3852)
Perm shape: (2000, 3852)
GT shape: (1000, 2)
Coord shape: (3852, 2)
Data loaded successfully!


In [26]:
# Step 3: 실제 데이터 처리 (NaiveDE + SpatialDE)
print("Processing real data...")

t0 = time.time()

# IMPORTANT: Calculate total_counts from ORIGINAL raw counts BEFORE any processing
counts_raw = counts.copy()  # Keep original raw counts
coord['total_counts'] = counts_raw.sum(axis=0)  # Sum across genes for each spot

print(f"Total counts calculated from raw data (min: {coord['total_counts'].min()}, max: {coord['total_counts'].max()})")


Processing real data...
Total counts calculated from raw data (min: 19512, max: 20561)


In [29]:

# NaiveDE normalization
counts = NaiveDE.stabilize(counts)
print(f"Normalized expression shape: {counts.shape}")

resid_expr = NaiveDE.regress_out(coord, counts, 'np.log(total_counts)').T
print(f"Residual expression shape: {resid_expr.shape}")

# SpatialDE analysis
X = coord[['X1', 'X2']].values
print(f"Coordinate matrix shape: {X.shape}")

ressig = SpatialDE.run(X, resid_expr)
t1 = time.time() - t0

print(f"SpatialDE completed in {t1:.2f} seconds")
print(f"Results shape: {ressig.shape}")
print(f"Significant genes (q < 0.05): {(ressig['qval'] < 0.05).sum()}")

Normalized expression shape: (2000, 3852)
Residual expression shape: (3852, 2000)
Coordinate matrix shape: (3852, 2)


Models: 100%|██████████| 10/10 [01:23<00:00,  8.31s/it]



SpatialDE completed in 1008.70 seconds
Results shape: (2000, 18)
Significant genes (q < 0.05): 1000


/home/lmseo/miniconda3/envs/sensim/lib/python3.9/site-packages/SpatialDE/base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
/home/lmseo/miniconda3/envs/sensim/lib/python3.9/site-packages/SpatialDE/util.py:19: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  pv = pv.ravel()  # flattens the array in place, more efficient than flatten()


In [30]:
# Step 4: 퍼뮤테이션 데이터 처리
print("Processing permuted data...")

# Run on permuted data
perm_t = perm.T
norm_expr_perm = NaiveDE.stabilize(perm_t.T).T
coord_perm = coord.copy()
coord_perm['total_counts'] = perm_t.sum(axis=1)
resid_expr_perm = NaiveDE.regress_out(coord_perm, norm_expr_perm.T, 'np.log(total_counts)').T

resperm = SpatialDE.run(X, resid_expr_perm)

print(f"Permutation results shape: {resperm.shape}")
print(f"Significant genes in permutation (q < 0.05): {(resperm['qval'] < 0.05).sum()}")

Processing permuted data...


Models: 100%|██████████| 10/10 [01:21<00:00,  8.11s/it]

Permutation results shape: (2000, 18)
Significant genes in permutation (q < 0.05): 0



/home/lmseo/miniconda3/envs/sensim/lib/python3.9/site-packages/SpatialDE/base.py:310: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  model_results = model_results[model_results.groupby(['g'])['max_ll'].transform(max) == model_results['max_ll']]
/home/lmseo/miniconda3/envs/sensim/lib/python3.9/site-packages/SpatialDE/util.py:19: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  pv = pv.ravel()  # flattens the array in place, more efficient than flatten()


In [31]:
# Step 5: 결과 계산 및 저장
print("Calculating final results...")

gs = g['gs'].values
gc = g['gc'].values
thr = 0.05

svgsim = ressig[ressig["qval"] < thr]["g"].values
svgprm = resperm[resperm["qval"] < thr]["g"].values

print(f"Ground truth signal genes: {len(gs)}")
print(f"Ground truth control genes: {len(gc)}")
print(f"Detected SVG in real data: {len(svgsim)}")
print(f"Detected SVG in permuted data: {len(svgprm)}")

# Calculate metrics
power_sim = len(np.intersect1d(gs, svgsim)) / len(gs) if len(gs) > 0 else 0
fpr_sim = len(np.intersect1d(gc, svgsim)) / len(gc) if len(gc) > 0 else 0
power_prm = len(np.intersect1d(gs, svgprm)) / len(gs) if len(gs) > 0 else 0
fpr_prm = len(np.intersect1d(gc, svgprm)) / len(gc) if len(gc) > 0 else 0

print(f"Power (real): {power_sim:.4f}")
print(f"FPR (real): {fpr_sim:.4f}")
print(f"Power (perm): {power_prm:.4f}")
print(f"FPR (perm): {fpr_prm:.4f}")

res = pd.DataFrame({
    'ID': ["Power_sim", "FPR_sim", "Power_prm", "FPR_prm"],
    'SPATIALDE': [power_sim, fpr_sim, power_prm, fpr_prm],
    'SI': [sit] * 4,
    'total_time': [t1] * 4,
    'mapping_time': [np.nan] * 4
})

output_file = os.path.join(rpath, f"res_spatialde_SI{sit}.csv")
res.to_csv(output_file, index=False)

print(f"Results saved to: {output_file}")
print("Final results:")
print(res)

Calculating final results...
Ground truth signal genes: 1000
Ground truth control genes: 1000
Detected SVG in real data: 1000
Detected SVG in permuted data: 0
Power (real): 1.0000
FPR (real): 0.0000
Power (perm): 0.0000
FPR (perm): 0.0000
Results saved to: ../data/bhmk.res/p/SPATIALDE/res_spatialde_SI0.47.csv
Final results:
          ID  SPATIALDE    SI  total_time  mapping_time
0  Power_sim        1.0  0.47   1008.7024           NaN
1    FPR_sim        0.0  0.47   1008.7024           NaN
2  Power_prm        0.0  0.47   1008.7024           NaN
3    FPR_prm        0.0  0.47   1008.7024           NaN


In [2]:
import numpy as np
si = np.linspace(0.1, 1.0, 30)

In [ ]:
si = np.ones(1)